# NB2 — VQA dùng-chung: **PhoBERT (câu hỏi) + ViT (ảnh)**, một model cho cả 10 câu

**Mô hình:** một model **duy nhất** cho toàn bộ 10 câu (multi-task). Đặc trưng hợp
nhất `x = [PhoBERT_meanpool(câu hỏi) ‖ ViT-L/14(ảnh)]` (768+1024 = 1792-d) → MLP →
**sigmoid trên GLOBAL ATOM VOCAB** (hợp nhất mọi nhãn nguyên tử của cả 10 câu).
Đây là baseline VQA "thật" theo công thức OpenViVQA/ViVQA (PhoBERT + ViT + hợp nhất).

**Điểm mấu chốt — chấm điểm theo ĐÚNG chuẩn per-question atomic (giống hệt nb1):**
- Câu *what* (Q2/Q5/Q8): decode = **ngưỡng** trên tập atom *ứng viên của câu đó* →
  exact-match, F1-macro/micro/samples, Recall-macro (đa nhãn nguyên tử).
- Câu nhị phân/đếm/không gian: decode = **argmax** trên atom ứng viên của câu → EM, F1-macro.

→ Không còn khung "combo-class toàn cục" cũ (đã bỏ). Mọi model (Prior, ViT+MLP, model
này) giờ **cùng một thước đo** nên xếp chung một bảng so sánh được.

> **Bài học LLaVA:** PhoBERT ở đây là **bộ mã hoá** (không sinh), câu hỏi ngắn cắt
> `max_length=64`, và chỉ có 10 câu hỏi cố định nên cache một lần → tuyệt đối không
> gặp lỗi tràn token như LLaVA (câu hỏi + lựa chọn dài vượt giới hạn, đáp án bị cắt).

Tham chiếu: Nguyen et al. 2023 (OpenViVQA); Tran et al. 2021 (ViVQA); PhoBERT
(Nguyen & Nguyen 2020); Lin et al. 2017 (Focal); Ben-Baruch et al. 2020 (ASL).

In [1]:
import os, sys, io, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
if os.path.basename(os.getcwd())!="experiments" and os.path.isdir("F:/VQA/experiments"):
    os.chdir("F:/VQA/experiments")
import unified as U
vit=U.load_vit("_vitl"); pho=U.load_phobert()
tr=pd.read_csv(os.path.join(U.QA,"train.csv")); tr["qid"]=tr["qid"].astype(str)
va=pd.read_csv(os.path.join(U.QA,"val.csv"));   va["qid"]=va["qid"].astype(str)
te=pd.read_csv(os.path.join(U.QA,"test.csv"));  te["qid"]=te["qid"].astype(str)
full=pd.concat([tr,va,te],ignore_index=True)
vocab,idx,cand=U.build_global_vocab(full)
Xtr,Ytr,qtr,rtr=U.build_rows(tr,vit,pho,idx,"full")
Xte,Yte,qte,rte=U.build_rows(te,vit,pho,idx,"full")
Xva,Yva,qva,rva=U.build_rows(va,vit,pho,idx,"full")
print("Đặc trưng hợp nhất:", Xtr.shape, "| GLOBAL atom vocab:", len(vocab))

Đặc trưng hợp nhất: (13672, 1792) | GLOBAL atom vocab: 20


## 1. EDA — GLOBAL atom vocab & tập atom ứng viên mỗi câu (cơ chế định tuyến)

Model học chung một không gian nhãn nguyên tử; PhoBERT giúp phân biệt câu nào dùng
tập atom nào. Bảng dưới cho thấy "đường ray" quyết định của từng câu.

In [2]:
print("GLOBAL atom vocab (%d):"%len(vocab)); print(vocab)
rows=[]
for q in U.QID_ORDER:
    rows.append({"qid":q,"loại":"multi" if q in U.WHAT_QIDS else "single",
                 "atom_ứng_viên":", ".join(vocab[i] for i in cand[q])})
pd.DataFrame(rows)

GLOBAL atom vocab (20):
['0', '1', '2', '3', '4', '5+', 'bục/bậc tự xây', 'chậu cây cá nhân', 'có', 'cửa hàng bày hàng ra hè', 'ghế/bàn cá nhân', 'hàng rong/sạp bán', 'không', 'không có', 'mái che/ô dù', 'quán ăn/quán nước/xe đẩy kinh doanh', 'sửa xe/rửa xe', 'xe máy', 'xe đạp', 'ô tô']


,qid,loại,atom_ứng_viên
0,Q1,single,"có, không"
1,Q2,multi,"cửa hàng bày hàng ra hè, hàng rong/sạp bán, kh..."
2,Q3,single,"có, không"
3,Q4,single,"có, không"
4,Q5,multi,"không có, xe máy, xe đạp, ô tô"
5,Q6,single,"0, 1, 2, 3, 4, 5+"
6,Q7,single,"có, không"
7,Q8,multi,"bục/bậc tự xây, chậu cây cá nhân, ghế/bàn cá n..."
8,Q9,single,"có, không"
9,Q10,single,"có, không"


## 2. Baseline — model đầy đủ (Q+I), chi tiết theo từng câu

In [3]:
t0=time.time()
model,sc,pos=U.train_shared(Xtr,Ytr,"baseline",epochs=80)
pred=U.predict_per_question(model,sc,Xte,qte,cand,"baseline")
t_base=time.time()-t0
w,s,_=U.score_tables(pred,Yte,qte,cand,vocab)
print("Baseline — what F1-macro=%.3f exact=%.3f | single F1-macro=%.3f EM=%.3f  (train+infer %.0fs)"%(
    w["F1_macro"],w["exact"],s["F1ma"],s["EM"],t_base))
U.per_question_breakdown(pred,Yte,qte,cand,vocab)

Baseline — what F1-macro=0.671 exact=0.712 | single F1-macro=0.705 EM=0.834  (train+infer 45s)


,qid,n,loại,EM/exact,F1_macro,R_macro
0,Q1,898,single,0.8575,0.8073,0.8073
1,Q2,577,multi,0.7643,0.7045,0.6732
2,Q3,412,single,0.8374,0.8367,0.8367
3,Q4,66,single,0.8788,0.6333,0.6333
4,Q5,559,multi,0.8122,0.6785,0.6418
5,Q6,399,single,0.6516,0.4681,0.4681
6,Q7,209,single,0.8947,0.7661,0.7661
7,Q8,324,multi,0.5586,0.6293,0.6293
8,Q9,179,single,0.8045,0.6629,0.6629
9,Q10,268,single,0.9104,0.7607,0.7607


## 3. Thực nghiệm nhiều phương pháp cân bằng ("A baseline" → "A + B")

Giữ nguyên kiến trúc PhoBERT+ViT dùng-chung, chỉ đổi **đúng một** yếu tố (hàm loss/
hiệu chỉnh/ngưỡng). Báo cáo cả hai nhóm câu theo chuẩn per-question atomic.

In [4]:
def label(m): return {"baseline":"PhoBERT+ViT-L (baseline)","weighted_bce":"+ Weighted-BCE",
        "focal":"+ Focal","asl":"+ ASL","logit_adjust":"+ Logit-Adjust","threshold":"+ Threshold-moving"}[m]
rows=[]; preds={}
for m in ["baseline","weighted_bce","focal","asl","logit_adjust","threshold"]:
    model,sc,pos=U.train_shared(Xtr,Ytr,m,epochs=80)
    p=U.predict_per_question(model,sc,Xte,qte,cand,m,pos_train=pos,Xval=Xva,Yval=Yva,qsval=qva)
    preds[m]=p; w,s,_=U.score_tables(p,Yte,qte,cand,vocab)
    rows.append({"mô_hình":label(m),
                 "what_exact":round(w["exact"],4),"what_F1macro":round(w["F1_macro"],4),
                 "what_F1micro":round(w["F1_micro"],4),"what_Rmacro":round(w["R_macro"],4),
                 "single_EM":round(s["EM"],4),"single_F1macro":round(s["F1ma"],4),"single_Rmacro":round(s["Rma"],4)})
shared_tbl=pd.DataFrame(rows); shared_tbl

,mô_hình,what_exact,what_F1macro,what_F1micro,what_Rmacro,single_EM,single_F1macro,single_Rmacro
0,PhoBERT+ViT-L (baseline),0.7099,0.6760,0.7741,0.6539,0.8317,0.7172,0.7091
1,+ Weighted-BCE,0.6428,0.6780,0.7602,0.7514,0.8297,0.7057,0.6950
2,+ Focal,0.7224,0.6757,0.7718,0.6414,0.8411,0.7301,0.7231
3,+ ASL,0.5063,0.6608,0.7229,0.8127,0.8222,0.6866,0.6927
4,+ Logit-Adjust,0.3638,0.5452,0.6370,0.9237,0.8167,0.7144,0.7073
5,+ Threshold-moving,0.6886,0.6759,0.7674,0.6916,0.8372,0.7353,0.7249


## 4. Ablation đầu vào — đóng góp của PhoBERT

Đổi đúng một yếu tố: có/không nhánh câu hỏi PhoBERT trong đầu vào (giữ nguyên cách
decode theo atom ứng viên của câu). So sánh cho biết đặc trưng câu hỏi PhoBERT đóng
góp thêm bao nhiêu ngoài việc đã biết tập atom ứng viên.

In [5]:
rows=[]
for mode in ["full","i_only"]:
    Xa,Ya,qa,_=U.build_rows(tr,vit,pho,idx,mode)
    Xb,Yb,qb,_=U.build_rows(te,vit,pho,idx,mode)
    model,sc,pos=U.train_shared(Xa,Ya,"baseline",epochs=80)
    p=U.predict_per_question(model,sc,Xb,qb,cand,"baseline")
    w,s,_=U.score_tables(p,Yb,qb,cand,vocab)
    rows.append({"đầu_vào":{"full":"Q+I (PhoBERT+ViT)","i_only":"I-only (bỏ PhoBERT)"}[mode],
                 "what_F1macro":round(w["F1_macro"],4),"single_F1macro":round(s["F1ma"],4),
                 "single_EM":round(s["EM"],4)})
abl=pd.DataFrame(rows); abl

,đầu_vào,what_F1macro,single_F1macro,single_EM
0,Q+I (PhoBERT+ViT),0.6682,0.7089,0.830
1,I-only (bỏ PhoBERT),0.5513,0.5264,0.591


## 5. Kiểm chứng 5-fold CV — baseline vs phương pháp tốt nhất

Gộp toàn bộ dữ liệu, chia 5-fold theo dòng. Báo cáo trung bình ± lệch (what F1-macro,
single F1-macro).

In [6]:
from sklearn.model_selection import KFold
Xall,Yall,qall,_=U.build_rows(full,vit,pho,idx,"full")
best_key = {v:k for k,v in {"baseline":"PhoBERT+ViT-L (baseline)","weighted_bce":"+ Weighted-BCE","focal":"+ Focal","asl":"+ ASL","logit_adjust":"+ Logit-Adjust","threshold":"+ Threshold-moving"}.items()}[
    shared_tbl.loc[shared_tbl["what_F1macro"].idxmax(),"mô_hình"]]
def cv_shared(method,folds=5):
    kf=KFold(n_splits=folds,shuffle=True,random_state=42); wf=[];sf=[]
    for trn,tst in kf.split(Xall):
        model,sc,pos=U.train_shared(Xall[trn],Yall[trn],method,epochs=80)
        p=U.predict_per_question(model,sc,Xall[tst],qall[tst],cand,method,pos_train=pos)
        w,s,_=U.score_tables(p,Yall[tst],qall[tst],cand,vocab); wf.append(w["F1_macro"]); sf.append(s["F1ma"])
    return np.mean(wf),np.std(wf),np.mean(sf),np.std(sf)
rows=[]
for m in dict.fromkeys(["baseline",best_key]):
    wm,ws,sm,ss=cv_shared(m)
    rows.append({"mô_hình":label(m),"what_F1macro_mean":round(wm,4),"what_F1macro_std":round(ws,4),
                 "single_F1macro_mean":round(sm,4),"single_F1macro_std":round(ss,4)})
cv_shared_df=pd.DataFrame(rows); cv_shared_df

,mô_hình,what_F1macro_mean,what_F1macro_std,single_F1macro_mean,single_F1macro_std
0,PhoBERT+ViT-L (baseline),0.6864,0.0101,0.7466,0.0112
1,+ Weighted-BCE,0.6761,0.0152,0.7422,0.0142


## 6. So sánh với NB1 (đầu riêng cho từng câu) — cùng một thước đo

Vì giờ **cùng chuẩn per-question atomic**, ta đặt thẳng model dùng-chung cạnh model
ViT+MLP có đầu riêng cho từng câu (NB1). Kỳ vọng: đầu riêng nhỉnh hơn một chút, nhưng
model dùng-chung là **một mô hình VQA multimodal thật** (có PhoBERT) — đánh đổi giữa
tính gọn (1 model) và điểm số.

In [7]:
try:
    nb1w=pd.read_csv("results/nb1_what_vitl.csv"); nb1s=pd.read_csv("results/nb1_single_vitl.csv")
    base_nb1_what=float(nb1w[nb1w["method"]=="baseline"]["F1_macro"])
    base_nb1_single=float(nb1s[nb1s["method"]=="baseline"]["F1ma"])
    cmp=pd.DataFrame([
      {"model":"ViT-L/14 + MLP (đầu riêng từng câu, NB1)","what_F1macro":round(base_nb1_what,4),"single_F1macro":round(base_nb1_single,4)},
      {"model":"PhoBERT + ViT-L dùng-chung (NB2)","what_F1macro":round(w["F1_macro"],4),"single_F1macro":round(s["F1ma"],4)},
    ])
    display(cmp)
except Exception as e:
    print("(chưa có kết quả NB1 để so sánh:",e,")")

,model,what_F1macro,single_F1macro
0,"ViT-L/14 + MLP (đầu riêng từng câu, NB1)",0.6952,0.5682
1,PhoBERT + ViT-L dùng-chung (NB2),0.5513,0.5264


## 7. Tổng hợp & lưu kết quả

In [8]:
os.makedirs("results",exist_ok=True)
shared_tbl.to_csv("results/nb2_shared_methods.csv",index=False)
abl.to_csv("results/nb2_input_ablation.csv",index=False)
cv_shared_df.to_csv("results/nb2_cv.csv",index=False)
U.per_question_breakdown(preds["baseline"],Yte,qte,cand,vocab).to_csv("results/nb2_per_question.csv",index=False)
print("Baseline what F1-macro   =", shared_tbl.iloc[0]["what_F1macro"])
print("Tốt nhất what F1-macro   =", shared_tbl["what_F1macro"].max(),
      "->", shared_tbl.loc[shared_tbl['what_F1macro'].idxmax(),'mô_hình'])
print("Baseline single F1-macro =", shared_tbl.iloc[0]["single_F1macro"])
print("Đã lưu 4 file results/nb2_*.csv")

Baseline what F1-macro   = 0.676
Tốt nhất what F1-macro   = 0.678 -> + Weighted-BCE
Baseline single F1-macro = 0.7172
Đã lưu 4 file results/nb2_*.csv
